# 01 — TSMOM Baseline

## <b>Block 01 — Data Audit</b>

### Objetivo
Verificar que el universo diario Gold está completo, consistente y alineado antes de calcular retornos o cualquier objeto de estrategia.

### Alcance
Universo: `SPY`, `QQQ`, `XLE`, `GLD`. Ventana solicitada: `2018-01-02` hasta la última sesión común disponible no posterior a `2026-11-17`.

### Fuente y causalidad
Este notebook lee exclusivamente `data/gold/daily_series/etf/<SYMBOL>.parquet`. No se permite look-ahead: las etapas posteriores deberán utilizar únicamente información disponible en cada instante de decisión.


In [91]:
from datetime import date
from pathlib import Path

import plotly.graph_objects as go
import polars as pl

SYMBOLS = ("SPY", "QQQ", "XLE", "GLD")
WINDOW_START = date(2018, 1, 2)
WINDOW_END = date(2026, 11, 17)

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        for project_root in (candidate, candidate / "Momentum_systematic_strategies"):
            if (project_root / "data" / "gold" / "daily_series" / "etf").is_dir():
                return project_root
    raise FileNotFoundError("No se encontró data/gold/daily_series/etf desde el directorio actual.")

PROJECT_ROOT = find_project_root(Path.cwd())
GOLD_ROOT = PROJECT_ROOT / "data" / "gold" / "daily_series" / "etf"
GOLD_ROOT


WindowsPath('C:/Users/avazq/OneDrive/Escritorio/cienciadatos/portafolio/quant-research-notes/Momentum_systematic_strategies/data/gold/daily_series/etf')

In [92]:
REQUIRED_COLUMNS = (
    "session_date", "open", "high", "low", "close", "adjusted_close",
    "volume", "dividend_amount", "split_coefficient",
)

frames = {}
for symbol in SYMBOLS:
    path = GOLD_ROOT / f"{symbol}.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Gold no disponible para {symbol}: {path}")
    frame = pl.read_parquet(path).with_columns(pl.col("session_date").cast(pl.Date))
    missing_columns = set(REQUIRED_COLUMNS).difference(frame.columns)
    if missing_columns:
        raise ValueError(f"{symbol} no contiene columnas requeridas: {sorted(missing_columns)}")
    frames[symbol] = frame.filter(
        pl.col("session_date").is_between(WINDOW_START, WINDOW_END, closed="both")
    ).sort("session_date")

{symbol: frame.shape for symbol, frame in frames.items()}


{'SPY': (2232, 14), 'QQQ': (2232, 14), 'XLE': (2232, 14), 'GLD': (2232, 14)}

In [93]:
def ohlc_violation_count(frame: pl.DataFrame) -> int:
    invalid_bar = (
        (pl.col("open") <= 0) | (pl.col("high") <= 0) |
        (pl.col("low") <= 0) | (pl.col("close") <= 0) |
        (pl.col("high") < pl.col("low")) |
        (pl.col("high") < pl.col("open")) |
        (pl.col("high") < pl.col("close")) |
        (pl.col("low") > pl.col("open")) |
        (pl.col("low") > pl.col("close"))
    )
    return frame.select(invalid_bar.sum()).item()

audit_rows = []
for symbol, frame in frames.items():
    null_counts = frame.select(pl.all().null_count()).to_dicts()[0]
    audit_rows.append({
        "symbol": symbol,
        "sessions": frame.height,
        "start": frame.get_column("session_date").min(),
        "end": frame.get_column("session_date").max(),
        "nulls_total": sum(null_counts.values()),
        "duplicate_dates": frame.select(pl.col("session_date").is_duplicated().sum()).item(),
        "ohlc_violations": ohlc_violation_count(frame),
        "adjusted_close_available": "adjusted_close" in frame.columns,
        "positive_dividend_sessions": frame.select((pl.col("dividend_amount") > 0).sum()).item(),
        "non_unit_split_sessions": frame.select((pl.col("split_coefficient") != 1).sum()).item(),
    })

asset_audit = pl.DataFrame(audit_rows)
asset_audit


symbol,sessions,start,end,nulls_total,duplicate_dates,ohlc_violations,adjusted_close_available,positive_dividend_sessions,non_unit_split_sessions
str,i64,date,date,i64,i64,i64,bool,i64,i64
"""SPY""",2232,2018-01-02,2026-11-17,0,0,0,true,35,0
"""QQQ""",2232,2018-01-02,2026-11-17,0,0,0,true,36,0
"""XLE""",2232,2018-01-02,2026-11-17,0,0,0,true,36,1
"""GLD""",2232,2018-01-02,2026-11-17,0,0,0,true,0,0


In [94]:
panel = None
for symbol in SYMBOLS:
    series = frames[symbol].select(
        "session_date", pl.col("adjusted_close").alias(symbol)
    )
    panel = series if panel is None else panel.join(series, on="session_date", how="inner")

panel = panel.sort("session_date")
panel_nulls = panel.select(pl.all().null_count()).to_dicts()[0]
panel_summary = {
    "rows": panel.height,
    "columns": panel.width,
    "start": panel.get_column("session_date").min(),
    "end": panel.get_column("session_date").max(),
    "nulls_total": sum(panel_nulls.values()),
    "duplicate_dates": panel.select(pl.col("session_date").is_duplicated().sum()).item(),
}

assert panel.columns == ["session_date", *SYMBOLS]
assert panel_summary["start"] == WINDOW_START
assert panel_summary["end"] <= WINDOW_END
assert panel_summary["nulls_total"] == 0

print(panel_summary)
panel.head()


{'rows': 2232, 'columns': 5, 'start': datetime.date(2018, 1, 2), 'end': datetime.date(2026, 11, 17), 'nulls_total': 0, 'duplicate_dates': 0}


session_date,SPY,QQQ,XLE,GLD
date,f64,f64,f64,f64
2018-01-02,235.20385,149.695763,25.380449,125.15
2018-01-03,236.69154,151.150312,25.760551,124.82
2018-01-04,237.689168,151.414775,25.916047,125.46
2018-01-05,239.273121,152.93544,25.905681,125.33
2018-01-08,239.710677,153.530483,26.061177,125.31


In [95]:
normalized_panel = panel.with_columns([
    (pl.col(symbol) / pl.col(symbol).first()).alias(symbol)
    for symbol in SYMBOLS
])

figure = go.Figure()
for symbol in SYMBOLS:
    figure.add_scatter(
        x=normalized_panel.get_column("session_date").to_list(),
        y=normalized_panel.get_column(symbol).to_list(),
        mode="lines",
        name=symbol,
    )

figure.update_layout(
    title="Normalized adjusted close (P_t / P_0)",
    xaxis_title="Session date",
    yaxis_title="P_t / P_0",
    template="plotly_white",
    legend_title="ETF",
)
figure.show()


### Resultados de la ejecución — 2026-11-17

| Activo | Sesiones | Rango | Nulos | Fechas duplicadas | Violaciones OHLC | `adjusted_close` | Dividendos positivos | Splits no unitarios |
| --- | ---: | --- | ---: | ---: | ---: | --- | ---: | ---: |
| SPY | 2,232 | 2018-01-02 → 2026-11-17 | 0 | 0 | 0 | Disponible | 35 | 0 |
| QQQ | 2,232 | 2018-01-02 → 2026-11-17 | 0 | 0 | 0 | Disponible | 36 | 0 |
| XLE | 2,232 | 2018-01-02 → 2026-11-17 | 0 | 0 | 0 | Disponible | 36 | 1 |
| GLD | 2,232 | 2018-01-02 → 2026-11-17 | 0 | 0 | 0 | Disponible | 0 | 0 |

La ventana común exacta es `2018-01-02 → 2026-11-17`. El panel wide `session_date | SPY | QQQ | XLE | GLD` tiene dimensión **2,232 × 5**, sin nulos ni fechas duplicadas. El split no unitario de XLE queda registrado como dato corporativo informativo; no constituye una violación OHLC.

Los datos quedan listos para continuar con Block 02.


## <b>Block 02 — Returns</b>

> Los retornos se calculan sobre el panel Gold limpio de Block 01. La medida principal es el retorno logarítmico, no el retorno porcentual simple.

$ r_t = \ln(P_t / P_{t-1}) $

Para cada horizonte `k`, el momentum continuo usa $ r_t^{(k)} = \ln(P_t / P_{t-k}) $. Todos estos valores usan exclusivamente precios conocidos hasta la fecha `t`; este bloque no usa retornos futuros ni crea posiciones.

In [96]:
MOMENTUM_HORIZONS = (21, 63, 126, 252)

return_columns = []
for symbol in SYMBOLS:
    log_price = pl.col(symbol).log()
    return_columns.append(log_price.diff().alias(f"{symbol}_ret_1d"))
    for horizon in MOMENTUM_HORIZONS:
        return_columns.append(
            (log_price - log_price.shift(horizon)).alias(f"{symbol}_mom_{horizon}")
        )

returns_panel = panel.with_columns(return_columns)
returns_panel.select("session_date", *[f"{symbol}_ret_1d" for symbol in SYMBOLS]).tail(3)


session_date,SPY_ret_1d,QQQ_ret_1d,XLE_ret_1d,GLD_ret_1d
date,f64,f64,f64,f64
2026-11-13,-0.003639,-0.012105,0.009746,0.001577
2026-11-16,-0.0193,-0.027043,-0.019744,-0.007115
2026-11-17,-0.010201,-0.00728,-0.005195,-0.001276


In [97]:
daily_return_audit = pl.DataFrame([
    {
        "asset": symbol,
        "mean daily return (%)": returns_panel.select(pl.col(f"{symbol}_ret_1d").mean() * 100).item(),
        "daily volatility (%)": returns_panel.select(pl.col(f"{symbol}_ret_1d").std() * 100).item(),
        "minimum daily return (%)": returns_panel.select(pl.col(f"{symbol}_ret_1d").min() * 100).item(),
        "maximum daily return (%)": returns_panel.select(pl.col(f"{symbol}_ret_1d").max() * 100).item(),
        "valid observations": returns_panel.select(pl.col(f"{symbol}_ret_1d").count()).item(),
    }
    for symbol in SYMBOLS
]).with_columns(pl.all().exclude(["asset", "valid observations"]).round(4))

daily_return_audit


asset,mean daily return (%),daily volatility (%),minimum daily return (%),maximum daily return (%),valid observations
str,f64,f64,f64,f64,i64
"""SPY""",0.0509,1.1993,-11.5887,9.9863,2231
"""QQQ""",0.0663,1.5068,-12.7592,11.3356,2231
"""XLE""",0.041,1.9723,-22.491,14.8742,2231
"""GLD""",0.0464,1.087,-10.8412,6.1647,2231


In [98]:
latest_horizons = pl.DataFrame([
    {
        "asset": symbol,
        **{
            f"{horizon}d momentum (%)": returns_panel.select(
                pl.col(f"{symbol}_mom_{horizon}").tail(1) * 100
            ).item()
            for horizon in MOMENTUM_HORIZONS
        },
    }
    for symbol in SYMBOLS
]).with_columns(pl.all().exclude("asset").round(4))

valid_observations = {
    horizon: returns_panel.select(pl.col(f"SPY_mom_{horizon}").count()).item()
    for horizon in MOMENTUM_HORIZONS
}

print(f"Latest date: {returns_panel.get_column('session_date').max()}")
print(f"Valid observations per horizon: {valid_observations}")
latest_horizons


Latest date: 2026-11-17
Valid observations per horizon: {21: 2211, 63: 2169, 126: 2106, 252: 1980}


asset,21d momentum (%),63d momentum (%),126d momentum (%),252d momentum (%)
str,f64,f64,f64,f64
"""SPY""",-4.5427,-4.6311,0.3263,9.6941
"""QQQ""",-8.9363,-8.5171,-6.3541,8.0661
"""XLE""",-6.7729,0.2604,4.638,34.7979
"""GLD""",-7.4361,-16.0486,-15.4815,-6.4489


### Información antes de discretizar

Los momentum returns son variables continuas: `+0.5%` y `+25%` contienen información de magnitud muy distinta. En Block 03, una señal clásica $ sign(r_t^{(k)}) $ convertiría ambas observaciones en `+1`; esa pérdida de información se documenta aquí, pero la señal no se implementa todavía.

In [99]:
spy_momentum = returns_panel.select(
    "session_date", *[f"SPY_mom_{horizon}" for horizon in MOMENTUM_HORIZONS]
)

figure = go.Figure()
for horizon in MOMENTUM_HORIZONS:
    figure.add_scatter(
        x=spy_momentum.get_column("session_date").to_list(),
        y=spy_momentum.get_column(f"SPY_mom_{horizon}").to_list(),
        mode="lines",
        name=f"{horizon}d",
    )

figure.add_hline(y=0, line_color="black", line_width=1)
figure.update_layout(
    title=(
        "SPY multi-horizon momentum returns"
        "<br><sup>Continuous momentum measures before signal discretization</sup>"
    ),
    xaxis_title="Session date",
    yaxis_title="Log momentum return",
    template="plotly_white",
    legend_title="Lookback",
)
figure.show()


### Resultados de la ejecución — 2026-11-17

| Activo | Media diaria | Volatilidad diaria | Mínimo diario | Máximo diario | Observaciones válidas |
| --- | ---: | ---: | ---: | ---: | ---: |
| SPY | 0.0509% | 1.1993% | -11.5887% | 9.9863% | 2,231 |
| QQQ | 0.0663% | 1.5068% | -12.7592% | 11.3356% | 2,231 |
| XLE | 0.0410% | 1.9723% | -22.4909% | 14.8742% | 2,231 |
| GLD | 0.0464% | 1.0870% | -10.8412% | 6.1647% | 2,231 |

Última fecha analizada: `2026-11-17`.

| Activo | 21d | 63d | 126d | 252d |
| --- | ---: | ---: | ---: | ---: |
| SPY | -4.5427% | -4.6311% | 0.3263% | 9.6941% |
| QQQ | -8.9363% | -8.5171% | -6.3541% | 8.0661% |
| XLE | -6.7729% | 0.2604% | 4.6380% | 34.7979% |
| GLD | -7.4361% | -16.0485% | -15.4815% | -6.4489% |

### <b>Block 02 conclusion</b>

- Los retornos diarios logarítmicos se calcularon para los cuatro activos; cada uno conserva 2,231 observaciones válidas.
- Los cuatro horizontes están disponibles: 21d (2,211 observaciones), 63d (2,169), 126d (2,106) y 252d (1,980).
- La pérdida por el lookback de 252 sesiones deja 1,980 observaciones útiles por activo.
- Los horizontes discrepan en signo y magnitud: al cierre, SPY es negativo a 21d y 63d, pero positivo a 126d y 252d.
- Las variables continuas están listas para construir señales TSMOM en el siguiente bloque.


## <b>Block 03 — TSMOM Signals</b>

> Las señales discretas se construyen únicamente a partir de los momentum returns continuos de Block 02.

La regla clásica es $ s_t^{(k)} = sign(r_t^{(k)}) $, con $ k \in \{21, 63, 126, 252\} $. Un retorno positivo produce `+1`, uno negativo `-1` y un retorno exactamente cero produce `0`; los valores iniciales sin lookback suficiente se mantienen nulos.

In [100]:
signal_columns = []
for symbol in SYMBOLS:
    for horizon in MOMENTUM_HORIZONS:
        momentum = pl.col(f"{symbol}_mom_{horizon}")
        signal_columns.append(
            pl.when(momentum.is_null()).then(None)
            .when(momentum > 0).then(1)
            .when(momentum < 0).then(-1)
            .otherwise(0)
            .alias(f"{symbol}_sig_{horizon}")
        )

signals_panel = returns_panel.with_columns(signal_columns)
signals_panel.select("session_date", *[f"SPY_sig_{horizon}" for horizon in MOMENTUM_HORIZONS]).tail(3)


session_date,SPY_sig_21,SPY_sig_63,SPY_sig_126,SPY_sig_252
date,i32,i32,i32,i32
2026-11-13,-1,-1,1,1
2026-11-16,-1,-1,1,1
2026-11-17,-1,-1,1,1


In [101]:
state_rows = []
for symbol in SYMBOLS:
    for horizon in MOMENTUM_HORIZONS:
        signal = pl.col(f"{symbol}_sig_{horizon}")
        valid = signals_panel.select(signal.count()).item()
        state_rows.append({
            "asset": symbol,
            "horizon": f"{horizon}d",
            "long (%)": signals_panel.select((signal == 1).sum() / valid * 100).item(),
            "short (%)": signals_panel.select((signal == -1).sum() / valid * 100).item(),
            "zero (%)": signals_panel.select((signal == 0).sum() / valid * 100).item(),
            "valid observations": valid,
        })

signal_state_summary = pl.DataFrame(state_rows).with_columns(
    pl.all().exclude(["asset", "horizon", "valid observations"]).round(2)
)
signal_state_summary


asset,horizon,long (%),short (%),zero (%),valid observations
str,str,f64,f64,f64,i64
"""SPY""","""21d""",67.8,32.2,0.0,2211
"""SPY""","""63d""",75.15,24.85,0.0,2169
"""SPY""","""126d""",78.44,21.56,0.0,2106
"""SPY""","""252d""",84.65,15.35,0.0,1980
"""QQQ""","""21d""",64.99,35.01,0.0,2211
…,…,…,…,…,…
"""XLE""","""252d""",65.45,34.55,0.0,1980
"""GLD""","""21d""",56.58,43.37,0.05,2211
"""GLD""","""63d""",65.24,34.76,0.0,2169


In [102]:
def run_lengths(values):
    values = [value for value in values if value is not None]
    lengths, current, length = [], values[0], 1
    for value in values[1:]:
        if value == current:
            length += 1
        else:
            lengths.append(length)
            current, length = value, 1
    return lengths + [length]

change_rows = []
for symbol in SYMBOLS:
    for horizon in MOMENTUM_HORIZONS:
        column = f"{symbol}_sig_{horizon}"
        previous = pl.col(column).shift(1)
        changes = signals_panel.select(
            ((pl.col(column) != previous) & pl.col(column).is_not_null() & previous.is_not_null()).sum()
        ).item()
        lengths = run_lengths(signals_panel.get_column(column).to_list())
        valid = signals_panel.select(pl.col(column).count()).item()
        change_rows.append({
            "asset": symbol,
            "horizon": f"{horizon}d",
            "signal changes": changes,
            "changes / year": changes / (valid / 252),
            "median streak (sessions)": pl.Series(lengths).median(),
            "mean streak (sessions)": pl.Series(lengths).mean(),
        })

signal_change_summary = pl.DataFrame(change_rows).with_columns(
    pl.all().exclude(["asset", "horizon", "signal changes"]).round(2)
)
signal_change_summary


asset,horizon,signal changes,changes / year,median streak (sessions),mean streak (sessions)
str,str,i64,f64,f64,f64
"""SPY""","""21d""",179,20.4,4.0,12.28
"""SPY""","""63d""",84,9.76,3.0,25.52
"""SPY""","""126d""",66,7.9,3.0,31.43
"""SPY""","""252d""",21,2.67,3.0,90.0
"""QQQ""","""21d""",179,20.4,3.0,12.28
…,…,…,…,…,…
"""XLE""","""252d""",51,6.49,6.0,38.08
"""GLD""","""21d""",231,26.33,3.0,9.53
"""GLD""","""63d""",75,8.71,4.0,28.54


Las siguientes correlaciones describen concordancia entre señales discretas de un mismo activo; no miden diversificación ni alpha.

In [103]:
correlation_matrices = {}
for symbol in SYMBOLS:
    columns = [f"{symbol}_sig_{horizon}" for horizon in MOMENTUM_HORIZONS]
    matrix_rows = []
    for left in columns:
        row = {"horizon": left.rsplit("_", 1)[-1] + "d"}
        for right in columns:
            row[right.rsplit("_", 1)[-1] + "d"] = signals_panel.select(
                pl.corr(pl.col(left), pl.col(right))
            ).item()
        matrix_rows.append(row)
    correlation_matrices[symbol] = pl.DataFrame(matrix_rows).with_columns(
        pl.all().exclude("horizon").round(3)
    )

for symbol, matrix in correlation_matrices.items():
    print(f"{symbol} signal correlation")
    print(matrix)


SPY signal correlation
shape: (4, 5)
┌─────────┬───────┬───────┬───────┬───────┐
│ horizon ┆ 21d   ┆ 63d   ┆ 126d  ┆ 252d  │
│ ---     ┆ ---   ┆ ---   ┆ ---   ┆ ---   │
│ str     ┆ f64   ┆ f64   ┆ f64   ┆ f64   │
╞═════════╪═══════╪═══════╪═══════╪═══════╡
│ 21d     ┆ 1.0   ┆ 0.452 ┆ 0.214 ┆ 0.187 │
│ 63d     ┆ 0.452 ┆ 1.0   ┆ 0.541 ┆ 0.388 │
│ 126d    ┆ 0.214 ┆ 0.541 ┆ 1.0   ┆ 0.54  │
│ 252d    ┆ 0.187 ┆ 0.388 ┆ 0.54  ┆ 1.0   │
└─────────┴───────┴───────┴───────┴───────┘
QQQ signal correlation
shape: (4, 5)
┌─────────┬───────┬───────┬───────┬───────┐
│ horizon ┆ 21d   ┆ 63d   ┆ 126d  ┆ 252d  │
│ ---     ┆ ---   ┆ ---   ┆ ---   ┆ ---   │
│ str     ┆ f64   ┆ f64   ┆ f64   ┆ f64   │
╞═════════╪═══════╪═══════╪═══════╪═══════╡
│ 21d     ┆ 1.0   ┆ 0.439 ┆ 0.274 ┆ 0.169 │
│ 63d     ┆ 0.439 ┆ 1.0   ┆ 0.545 ┆ 0.336 │
│ 126d    ┆ 0.274 ┆ 0.545 ┆ 1.0   ┆ 0.656 │
│ 252d    ┆ 0.169 ┆ 0.336 ┆ 0.656 ┆ 1.0   │
└─────────┴───────┴───────┴───────┴───────┘
XLE signal correlation
shape: (4, 5)
┌────────

In [104]:
spy_disagreements = signals_panel.filter(
    (pl.col("SPY_sig_21") != pl.col("SPY_sig_252"))
    & pl.col("SPY_sig_21").is_not_null()
    & pl.col("SPY_sig_252").is_not_null()
).select(
    "session_date",
    pl.col("SPY").alias("price"),
    (pl.col("SPY_mom_21") * 100).round(4).alias("mom_21 (%)"),
    pl.col("SPY_sig_21").alias("sig_21"),
    (pl.col("SPY_mom_252") * 100).round(4).alias("mom_252 (%)"),
    pl.col("SPY_sig_252").alias("sig_252"),
).tail(10)

spy_disagreements


session_date,price,mom_21 (%),sig_21,mom_252 (%),sig_252
date,f64,f64,i32,f64,i32
2026-11-04,766.81,-1.5888,-1,12.5961,1
2026-11-05,733.0,-5.8578,-1,9.2792,1
2026-11-06,735.29,-5.1216,-1,9.2451,1
2026-11-09,752.69,-3.3805,-1,12.6627,1
2026-11-10,749.92,-3.6889,-1,12.1956,1
2026-11-11,759.81,-2.3671,-1,11.9574,1
2026-11-12,757.12,-1.6349,-1,11.3741,1
2026-11-13,754.37,-2.283,-1,10.9546,1
2026-11-16,739.95,-3.1003,-1,10.6979,1


In [105]:
from plotly.subplots import make_subplots

figure = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    row_heights=[0.7, 0.3],
)
figure.add_scatter(
    x=signals_panel.get_column("session_date").to_list(),
    y=signals_panel.get_column("SPY").to_list(),
    mode="lines", name="SPY adjusted close", line=dict(color="black"), row=1, col=1,
)
for horizon, color in ((21, "#d62728"), (252, "#1f77b4")):
    figure.add_scatter(
        x=signals_panel.get_column("session_date").to_list(),
        y=signals_panel.get_column(f"SPY_sig_{horizon}").to_list(),
        mode="lines", line_shape="hv", name=f"{horizon}d signal",
        line=dict(color=color), row=2, col=1,
    )

figure.update_layout(
    title="SPY — short vs long horizon TSMOM signals",
    template="plotly_white", height=620, legend_title="Series",
)
figure.update_yaxes(title_text="Adjusted close", row=1, col=1)
figure.update_yaxes(
    title_text="Signal", tickvals=[-1, 0, 1], ticktext=["-1", "0", "+1"], row=2, col=1,
)
figure.show()


In [106]:
latest_signals = pl.DataFrame([
    {
        "symbol": symbol,
        **{f"{horizon}d": signals_panel.select(pl.col(f"{symbol}_sig_{horizon}").tail(1)).item()
           for horizon in MOMENTUM_HORIZONS},
    }
    for symbol in SYMBOLS
])

latest_signals


symbol,21d,63d,126d,252d
str,i64,i64,i64,i64
"""SPY""",-1,-1,1,1
"""QQQ""",-1,-1,-1,1
"""XLE""",-1,1,1,1
"""GLD""",-1,-1,-1,-1


### Información retenida por la señal

La señal clásica conserva únicamente dirección: `+1`, `0` o `-1`. Por tanto, `+0.5%` y `+25%` generan la misma señal `+1`. La regla es interpretable, pero elimina información sobre magnitud y confianza.

### <b>Block 03 conclusion</b>

- Los estados long predominan en los cuatro activos y horizontes; los estados `0` sólo aparecen en dos observaciones de GLD.
- En SPY, 21d cambia aproximadamente 20.4 veces por año, frente a 2.7 cambios por año en 252d.
- La racha media de SPY dura 12.3 sesiones a 21d y 90.0 sesiones a 252d.
- El acuerdo entre 21d y 252d es bajo a moderado: su correlación va de 0.147 (GLD) a 0.187 (SPY).
- Los desacuerdos son visibles al cierre: SPY está short a 21d y long a 252d.

Las señales quedan preparadas para estudiar su realización económica en Block 04.


## <b>Block 04 — Causal PnL</b>

> La señal conocida al cierre de hoy se aplica sólo al retorno realizado de la siguiente sesión.

La regla es $ R_{t+1}^{strategy,(k)} = s_t^{(k)} \cdot r_{t+1} $. El `shift(-1)` deja visible que la señal de la fecha `t` no opera sobre el retorno de esa misma fecha, evitando look-ahead.

In [107]:
pnl_columns = []
for symbol in SYMBOLS:
    next_return = pl.col(f"{symbol}_ret_1d").shift(-1)
    pnl_columns.append(next_return.alias(f"{symbol}_next_ret"))
    for horizon in MOMENTUM_HORIZONS:
        pnl_columns.append(
            (pl.col(f"{symbol}_sig_{horizon}") * next_return).alias(f"{symbol}_strat_{horizon}")
        )

pnl_panel = signals_panel.with_columns(pnl_columns)
pnl_panel.select("session_date", "SPY_sig_21", "SPY_next_ret", "SPY_strat_21").tail(3)


session_date,SPY_sig_21,SPY_next_ret,SPY_strat_21
date,i32,f64,f64
2026-11-13,-1,-0.0193,0.0193
2026-11-16,-1,-0.010201,0.010201
2026-11-17,-1,null,null


In [108]:
causal_check = pnl_panel.select(
    "session_date",
    pl.col("SPY_sig_21").alias("signal_t"),
    pl.col("SPY_next_ret").round(4).alias("next_return"),
    pl.col("SPY_strat_21").round(4).alias("strategy_return_t_plus_1"),
).drop_nulls().head(5)

causal_check


session_date,signal_t,next_return,strategy_return_t_plus_1
date,i32,f64,f64
2018-02-01,1,-0.022,-0.022
2018-02-02,1,-0.0427,-0.0427
2018-02-05,-1,0.0195,-0.0195
2018-02-06,-1,-0.0054,0.0054
2018-02-07,-1,-0.0382,0.0382


Para comparar los cuatro lookbacks directamente, las métricas usan la ventana común que comienza cuando 252d ya está disponible y termina antes de la última sesión sin retorno siguiente. Buy & Hold usa exactamente esa misma muestra para cada activo.

In [109]:
strategy_columns = [
    f"{symbol}_strat_{horizon}"
    for symbol in SYMBOLS
    for horizon in MOMENTUM_HORIZONS
]
next_return_columns = [f"{symbol}_next_ret" for symbol in SYMBOLS]
comparison_panel = pnl_panel.drop_nulls(strategy_columns + next_return_columns)

comparison_window = {
    "start": comparison_panel.get_column("session_date").min(),
    "end": comparison_panel.get_column("session_date").max(),
    "observations": comparison_panel.height,
}
comparison_window


{'start': datetime.date(2019, 1, 3),
 'end': datetime.date(2026, 11, 16),
 'observations': 1979}

In [110]:
def performance_metrics(log_returns):
    values = [value for value in log_returns if value is not None]
    series = pl.Series(values)
    wealth = series.cum_sum().exp()
    drawdown = wealth / wealth.cum_max() - 1
    daily_volatility = series.std()
    return {
        "cumulative return (%)": (wealth[-1] - 1) * 100,
        "annualized return (%)": (wealth[-1] ** (252 / len(series)) - 1) * 100,
        "annualized vol (%)": daily_volatility * (252 ** 0.5) * 100,
        "Sharpe (rf=0)": series.mean() / daily_volatility * (252 ** 0.5),
        "max drawdown (%)": drawdown.min() * 100,
    }

performance_rows = []
for symbol in SYMBOLS:
    performance_rows.append({
        "asset": symbol,
        "strategy": "Buy & Hold",
        **performance_metrics(comparison_panel.get_column(f"{symbol}_next_ret").to_list()),
    })
    for horizon in MOMENTUM_HORIZONS:
        performance_rows.append({
            "asset": symbol,
            "strategy": f"TSMOM {horizon}d",
            **performance_metrics(comparison_panel.get_column(f"{symbol}_strat_{horizon}").to_list()),
        })

performance_summary = pl.DataFrame(performance_rows).with_columns(
    pl.all().exclude(["asset", "strategy"]).round(3)
)
performance_summary


asset,strategy,cumulative return (%),annualized return (%),annualized vol (%),Sharpe (rf=0),max drawdown (%)
str,str,f64,f64,f64,f64,f64
"""SPY""","""Buy & Hold""",236.294,16.7,19.258,0.802,-33.7
"""SPY""","""TSMOM 21d""",45.135,4.858,19.28,0.246,-35.646
"""SPY""","""TSMOM 63d""",30.415,3.439,19.281,0.175,-34.599
"""SPY""","""TSMOM 126d""",1.097,0.139,19.282,0.007,-39.254
"""SPY""","""TSMOM 252d""",28.682,3.263,19.281,0.167,-43.423
…,…,…,…,…,…,…
"""GLD""","""Buy & Hold""",187.903,14.414,17.991,0.748,-28.921
"""GLD""","""TSMOM 21d""",26.828,3.073,18.006,0.168,-42.112
"""GLD""","""TSMOM 63d""",122.516,10.722,17.999,0.566,-36.47


In [111]:
cross_asset_summary = performance_summary.filter(
    pl.col("strategy") != "Buy & Hold"
).select(
    "asset", "strategy", "annualized return (%)", "annualized vol (%)",
    "Sharpe (rf=0)", "max drawdown (%)",
)

cross_asset_summary


asset,strategy,annualized return (%),annualized vol (%),Sharpe (rf=0),max drawdown (%)
str,str,f64,f64,f64,f64
"""SPY""","""TSMOM 21d""",4.858,19.28,0.246,-35.646
"""SPY""","""TSMOM 63d""",3.439,19.281,0.175,-34.599
"""SPY""","""TSMOM 126d""",0.139,19.282,0.007,-39.254
"""SPY""","""TSMOM 252d""",3.263,19.281,0.167,-43.423
"""QQQ""","""TSMOM 21d""",6.511,24.053,0.262,-41.39
…,…,…,…,…,…
"""XLE""","""TSMOM 252d""",10.227,32.28,0.302,-55.564
"""GLD""","""TSMOM 21d""",3.073,18.006,0.168,-42.112
"""GLD""","""TSMOM 63d""",10.722,17.999,0.566,-36.47


In [112]:
spy_returns = {
    "Buy & Hold": "SPY_next_ret",
    **{f"TSMOM {horizon}d": f"SPY_strat_{horizon}" for horizon in MOMENTUM_HORIZONS},
}
spy_wealth = comparison_panel.select(
    "session_date",
    *[pl.col(column).cum_sum().exp().alias(name) for name, column in spy_returns.items()],
)

figure = go.Figure()
for name in spy_returns:
    figure.add_scatter(
        x=spy_wealth.get_column("session_date").to_list(),
        y=spy_wealth.get_column(name).to_list(),
        mode="lines",
        name=name,
    )

figure.update_layout(
    title="SPY — causal TSMOM performance by lookback",
    xaxis_title="Session date",
    yaxis_title="Cumulative growth (initial value = 1)",
    template="plotly_white",
    legend_title="Strategy",
)
figure.show()


In [113]:
drawdown_returns = {
    "Buy & Hold": "SPY_next_ret",
    "TSMOM 21d": "SPY_strat_21",
    "TSMOM 252d": "SPY_strat_252",
}
spy_drawdowns = comparison_panel.select(
    "session_date",
    *[
        (pl.col(column).cum_sum().exp() / pl.col(column).cum_sum().exp().cum_max() - 1).alias(name)
        for name, column in drawdown_returns.items()
    ],
)

figure = go.Figure()
for name in drawdown_returns:
    figure.add_scatter(
        x=spy_drawdowns.get_column("session_date").to_list(),
        y=spy_drawdowns.get_column(name).to_list(),
        mode="lines",
        name=name,
    )

figure.update_layout(
    title="SPY — drawdown comparison",
    xaxis_title="Session date",
    yaxis_title="Drawdown",
    template="plotly_white",
    legend_title="Strategy",
)
figure.add_hline(y=0, line_color="black", line_width=1)
figure.show()


### Observaciones

- La volatilidad anualizada cambia muy poco entre horizontes de un mismo activo porque la regla mantiene exposición absoluta cercana a uno; XLE conserva la mayor volatilidad y GLD la menor.
- Los drawdowns sí difieren: XLE 126d alcanza aproximadamente -75.4%, mientras que SPY 252d alcanza -43.4%, por debajo de su Buy & Hold de -33.7%.
- La señal 21d cambia con mucha mayor frecuencia, pero eso todavía no incorpora costes de rotación ni fricciones de ejecución.
- Hay variación material entre activos: XLE 126d tiene retorno anualizado negativo en esta ventana, mientras que GLD 63d y 126d conservan retornos anualizados superiores a 10%.
- Buy & Hold supera claramente a las variantes TSMOM de SPY y QQQ en esta muestra; esto es una descripción histórica, no una selección de horizonte.


### <b>04 conclusion</b>

- La alineación causal usa la señal al cierre de `t` con el retorno logarítmico de `t+1`; la muestra común contiene 1,979 observaciones.
- Return, volatilidad y drawdown cambian de forma apreciable según horizonte, especialmente en XLE y GLD.
- La señal rápida genera más cambios de estado, mientras que la lenta concentra rachas más largas y drawdowns distintos.
- Los resultados varían de forma material entre activos y Buy & Hold domina algunos tramos de la muestra.
- No se han incorporado costes, financiación, sizing ni ejecución realista.

El siguiente paso será incorporar una capa separada de ejecución y costes, reutilizando selectivamente componentes del notebook de volatility targeting.


## <b>Block 05 — Diagnostics</b>

> Estos diagnósticos describen cómo las patas long y short, los cambios de señal y las rachas contribuyen al PnL causal de Block 04. No cambian la definición original de TSMOM.


### <b>Long/short y long/cash</b>

La variante long/cash conserva el retorno siguiente solo cuando la señal es $+1$; con señal $-1$ o $0$, asigna retorno cero. Se utiliza únicamente como contraste descriptivo.


In [114]:
long_cash_columns = []

for symbol in SYMBOLS:
    for horizon in MOMENTUM_HORIZONS:
        signal = pl.col(f"{symbol}_signal_{horizon}")
        next_return = pl.col(f"{symbol}_next_ret")
        long_cash_columns.append(
            pl.when(signal.is_null())
            .then(None)
            .when(signal == 1)
            .then(next_return)
            .otherwise(0.0)
            .alias(f"{symbol}_long_cash_{horizon}")
        )

diagnostic_panel = pnl_panel.with_columns(long_cash_columns).drop_nulls(
    strategy_columns + next_return_columns
)
diagnostic_panel.select("session_date", "SPY_strat_21", "SPY_long_cash_21").head()


ColumnNotFoundError: unable to find column "SPY_signal_21"; valid columns: ["session_date", "SPY", "QQQ", "XLE", "GLD", "SPY_ret_1d", "SPY_mom_21", "SPY_mom_63", "SPY_mom_126", "SPY_mom_252", "QQQ_ret_1d", "QQQ_mom_21", "QQQ_mom_63", "QQQ_mom_126", "QQQ_mom_252", "XLE_ret_1d", "XLE_mom_21", "XLE_mom_63", "XLE_mom_126", "XLE_mom_252", "GLD_ret_1d", "GLD_mom_21", "GLD_mom_63", "GLD_mom_126", "GLD_mom_252", "SPY_sig_21", "SPY_sig_63", "SPY_sig_126", "SPY_sig_252", "QQQ_sig_21", "QQQ_sig_63", "QQQ_sig_126", "QQQ_sig_252", "XLE_sig_21", "XLE_sig_63", "XLE_sig_126", "XLE_sig_252", "GLD_sig_21", "GLD_sig_63", "GLD_sig_126", "GLD_sig_252", "SPY_next_ret", "SPY_strat_21", "SPY_strat_63", "SPY_strat_126", "SPY_strat_252", "QQQ_next_ret", "QQQ_strat_21", "QQQ_strat_63", "QQQ_strat_126", "QQQ_strat_252", "XLE_next_ret", "XLE_strat_21", "XLE_strat_63", "XLE_strat_126", "XLE_strat_252", "GLD_next_ret", "GLD_strat_21", "GLD_strat_63", "GLD_strat_126", "GLD_strat_252"]

Did you mean "SPY_sig_21"?

In [ ]:
leg_rows = []

for symbol in SYMBOLS:
    for horizon in MOMENTUM_HORIZONS:
        signal_column = f"{symbol}_signal_{horizon}"
        strategy_column = f"{symbol}_strat_{horizon}"
        for state, leg in [(1, "long"), (-1, "short"), (0, "flat")]:
            returns = diagnostic_panel.filter(pl.col(signal_column) == state).get_column(strategy_column)
            if len(returns):
                leg_rows.append({
                    "asset": symbol,
                    "horizon": f"{horizon}d",
                    "leg": leg,
                    "observations": len(returns),
                    "cumulative log contribution (%)": returns.sum() * 100,
                    "mean daily return (%)": returns.mean() * 100,
                    "positive days (%)": (returns.gt(0).sum() / len(returns)) * 100,
                })

leg_summary = pl.DataFrame(leg_rows).with_columns(
    pl.col([
        "cumulative log contribution (%)", "mean daily return (%)", "positive days (%)"
    ]).round(3)
)
leg_summary


La contribución acumulada se expresa como suma de retornos logarítmicos, por lo que las patas son aditivas dentro de cada estrategia.


In [ ]:
variant_rows = []

for symbol in SYMBOLS:
    buy_hold = performance_metrics(diagnostic_panel.get_column(f"{symbol}_next_ret").to_list())
    for horizon in MOMENTUM_HORIZONS:
        for variant, column in [
            ("TSMOM long/short", f"{symbol}_strat_{horizon}"),
            ("TSMOM long/cash", f"{symbol}_long_cash_{horizon}"),
        ]:
            metrics = performance_metrics(diagnostic_panel.get_column(column).to_list())
            variant_rows.append({
                "asset": symbol, "horizon": f"{horizon}d", "variant": variant, **metrics
            })
    variant_rows.append({
        "asset": symbol, "horizon": "Buy & Hold", "variant": "Buy & Hold", **buy_hold
    })

variant_summary = pl.DataFrame(variant_rows).select(
    "asset", "horizon", "variant", "annualized return (%)", "annualized vol (%)",
    "Sharpe (rf=0)", "max drawdown (%)",
).with_columns(
    pl.col(["annualized return (%)", "annualized vol (%)", "Sharpe (rf=0)", "max drawdown (%)"]).round(3)
)
variant_summary


### <b>SPY por año</b>

> 2026 contiene solo las sesiones disponibles en la muestra.


In [ ]:
spy_yearly = (
    diagnostic_panel
    .with_columns(pl.col("session_date").dt.year().alias("year"))
    .group_by("year")
    .agg([
        ((pl.col(column).sum().exp() - 1) * 100).alias(name)
        for name, column in {
            "Buy & Hold (%)": "SPY_next_ret",
            "TSMOM 21d (%)": "SPY_strat_21",
            "TSMOM 252d (%)": "SPY_strat_252",
            "Long/cash 21d (%)": "SPY_long_cash_21",
            "Long/cash 252d (%)": "SPY_long_cash_252",
        }.items()
    ])
    .sort("year")
    .with_columns(pl.all().exclude("year").round(3))
)
spy_yearly


In [ ]:
figure = go.Figure()
for column in ["Buy & Hold (%)", "TSMOM 21d (%)", "TSMOM 252d (%)"]:
    figure.add_bar(name=column.removesuffix(" (%)"), x=spy_yearly["year"], y=spy_yearly[column])

figure.update_layout(
    title="SPY — annual Buy & Hold and causal TSMOM returns",
    barmode="group",
    yaxis_title="return (%)",
    legend_title_text="",
)
figure.show()


### <b>Whipsaw y rachas de SPY</b>

Un cambio se define como $s_t \neq s_{t-1}$. Los retornos posteriores se calculan sobre la estrategia causal ya alineada en Block 04.


In [ ]:
import math

whipsaw_rows = []
for horizon in (21, 252):
    signals = diagnostic_panel.get_column(f"SPY_signal_{horizon}").to_list()
    returns = diagnostic_panel.get_column(f"SPY_strat_{horizon}").to_list()
    changes = [index for index in range(1, len(signals)) if signals[index] != signals[index - 1]]

    for sessions in (1, 5, 10):
        outcomes = [math.exp(sum(returns[index:index + sessions])) - 1 for index in changes]
        whipsaw_rows.append({
            "horizon": f"{horizon}d",
            "sessions after change": sessions,
            "changes": len(changes),
            "mean return (%)": pl.Series(outcomes).mean() * 100,
            "median return (%)": pl.Series(outcomes).median() * 100,
            "negative outcomes (%)": pl.Series(outcomes).lt(0).mean() * 100,
        })

whipsaw_summary = pl.DataFrame(whipsaw_rows).with_columns(
    pl.col(["mean return (%)", "median return (%)", "negative outcomes (%)"]).round(3)
)
whipsaw_summary


In [ ]:
run_rows = []
for horizon in (21, 252):
    signals = diagnostic_panel.get_column(f"SPY_signal_{horizon}").to_list()
    returns = diagnostic_panel.get_column(f"SPY_strat_{horizon}").to_list()
    starts = [0] + [index for index in range(1, len(signals)) if signals[index] != signals[index - 1]]
    ends = starts[1:] + [len(signals)]
    run_returns = [math.exp(sum(returns[start:end])) - 1 for start, end in zip(starts, ends)]
    run_series = pl.Series(run_returns)
    run_rows.append({
        "horizon": f"{horizon}d",
        "runs": len(run_returns),
        "winning runs (%)": run_series.gt(0).mean() * 100,
        "median PnL (%)": run_series.median() * 100,
        "best run (%)": run_series.max() * 100,
        "worst run (%)": run_series.min() * 100,
    })

run_summary = pl.DataFrame(run_rows).with_columns(
    pl.col(["winning runs (%)", "median PnL (%)", "best run (%)", "worst run (%)"]).round(3)
)
run_summary


In [ ]:
spy_diagnostic_wealth = diagnostic_panel.select(
    "session_date",
    pl.col("SPY_next_ret").cum_sum().exp().alias("Buy & Hold"),
    pl.col("SPY_strat_21").cum_sum().exp().alias("TSMOM 21d long/short"),
    pl.col("SPY_long_cash_21").cum_sum().exp().alias("TSMOM 21d long/cash"),
)

figure = go.Figure()
for column in spy_diagnostic_wealth.columns[1:]:
    figure.add_scatter(
        x=spy_diagnostic_wealth["session_date"], y=spy_diagnostic_wealth[column], mode="lines", name=column
    )

figure.update_layout(
    title="SPY — effect of removing the short leg",
    yaxis_title="growth of 1",
    legend_title_text="",
)
figure.show()


### <b>Interpretación</b>

- En SPY, las patas long aportan contribuciones acumuladas positivas en los cuatro horizontes, mientras que las short restan en esta muestra; por ejemplo, en 21d son aproximadamente +79.3% y -42.0% en retornos logarítmicos acumulados.
- El contraste 21d cambia de 4.9% anualizado long/short a 10.6% long/cash, con menor volatilidad y drawdown. Es evidencia descriptiva de que la pata short pesa en el resultado observado, no una elección de variante.
- En 2022, TSMOM 21d supera a Buy & Hold de SPY; en 2021 queda claramente por detrás. El horizonte 252d tiene su mayor diferencia negativa relativa en 2020.
- Los cambios de 21d son mucho más frecuentes que los de 252d. En esta muestra, sus retornos posteriores medios no son sistemáticamente peores; los pocos cambios de 252d presentan medias negativas en las tres ventanas.
- Solo 35.7% de las rachas de 21d y 27.3% de las de 252d son ganadoras; las rachas positivas grandes conviven con una mediana negativa.


### <b>Conclusión</b>

- La pata short explica una parte importante del deterioro de SPY en esta ventana; la contribución long permanece positiva.
- Long/cash mejora la geometría de retorno, volatilidad y drawdown frente a long/short para varios activos, sin definir todavía una variante definitiva.
- El aporte de momentum es heterogéneo por año y por activo; no hay un horizonte único que domine toda la muestra.
- La mayor frecuencia de cambios de 21d no basta, por sí sola, para atribuirle pérdidas posteriores más repetidas que a 252d.
- Las rachas muestran pocas observaciones ganadoras y resultados dispersos, compatibles con una distribución concentrada en movimientos grandes.

El siguiente experimento lógico es estudiar cómo cambian estas señales al introducir volatility targeting, sin asumir que necesariamente mejorará el resultado.
